# Final Test Locked

Pipeline finale locked. Il validation seleziona; il test valuta. Il dry-run non carica modelli e non scrive artefatti scientifici.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "configs/final_classifier_registry.json").is_file():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/utility"))
from final_classifier_evaluation import *

DRY_RUN = True
RECOMPUTE_TEST_PREDICTIONS = False
ALLOW_UNVERIFIED_LEGACY_PREDICTIONS = False
TEST_BATCH_SIZE = 8
TEST_NUM_WORKERS = 4
DEVICE = "auto"
PATIENT_AGGREGATION = "mean"
TEST_CSV = PROJECT_ROOT / "data/processed/metadata/test.csv"
TEST_DATASET_MANIFEST = PROJECT_ROOT / "results/final_evaluation/test_dataset_manifest.json"
REGISTRY_PATH = PROJECT_ROOT / "configs/final_classifier_registry.json"

DRY_RUN = True
MANIFEST_PATH = PROJECT_ROOT / "results/final_evaluation/finalists_manifest.json"
OUTPUT_DIR = PROJECT_ROOT / "results/final_evaluation"
CENTRAL_PREDICTIONS = OUTPUT_DIR / "test_predictions"

In [ ]:
# Scientific-only read: this must succeed and list blockers/missing predictions even while
# the lock is operationally incomplete (e.g. Mammo-FM/ResNet blockers) -- it must not crash here.
manifest = validate_locked_finalists_manifest(MANIFEST_PATH, require_operational_complete=False)
registry = {x["experiment_id"]: x for x in build_experiment_registry(REGISTRY_PATH)}
missing = []
for finalist in manifest["finalists"]:
    exp = registry[finalist["experiment_id"]]
    paths = canonical_test_prediction_paths(exp)
    pred = PROJECT_ROOT / paths["test_predictions_path"]
    if not pred.is_file(): missing.append({"experiment_id": exp["experiment_id"], "notebook": exp.get("test_notebook"), "expected": str(pred)})
print("Finalisti locked:", [x["experiment_id"] for x in manifest["finalists"]])
print("scientific_selection_complete:", manifest.get("scientific_selection_complete"), "| final_aggregation_complete:", manifest.get("final_aggregation_complete"))
print("Blocker operativi:", manifest.get("operational_blockers")); print("Predizioni mancanti:", missing)
if DRY_RUN: print("DRY_RUN: nessuna predizione copiata, nessuna metrica test scritta.")

In [ ]:
if not DRY_RUN:
    # Operational gate: only proceed once every finalist has real, verified test predictions.
    manifest = validate_locked_finalists_manifest(MANIFEST_PATH, require_operational_complete=True)
    if missing: raise RuntimeError(f"Copertura test incompleta: {missing}")
    canonical = pd.read_csv(TEST_CSV); canonical_ids = canonical.patient_id.astype(str)
    CENTRAL_PREDICTIONS.mkdir(parents=True, exist_ok=True); frames = {}; source_paths = {}; metric_rows = []
    for finalist in manifest["finalists"]:
        exp = registry[finalist["experiment_id"]]
        paths = canonical_test_prediction_paths(exp)
        source = PROJECT_ROOT / paths["test_predictions_path"]
        source_manifest = PROJECT_ROOT / paths["test_predictions_manifest_path"]
        if not source_manifest.is_file(): raise RuntimeError(f"Manifest sorgente mancante: {source_manifest}")
        source_payload = json.loads(source_manifest.read_text())
        required = {"experiment_id": exp["experiment_id"], "checkpoint_signature": finalist["checkpoint_signature"], "validation_metrics_signature": content_signature(PROJECT_ROOT / exp["validation_metrics_path"]), "validation_threshold": finalist["validation_threshold"], "threshold_method": finalist["threshold_method"], "test_dataset_manifest_signature": content_signature(TEST_DATASET_MANIFEST), "patient_ids_hash": patient_ids_hash(canonical_ids), "test_used_for_selection": False}
        incompatible = [key for key, value in required.items() if source_payload.get(key) != strict_jsonable(value)]
        if source_payload.get("prediction_file_signature") != content_signature(source): incompatible.append("prediction_file_signature")
        level = source_payload.get("provenance_level", "invalid")
        if level == "legacy_normalized_unverified" and not ALLOW_UNVERIFIED_LEGACY_PREDICTIONS: incompatible.append("provenance_level")
        if level not in {"verified_native", "verified_recomputed", "legacy_normalized_unverified"}: incompatible.append("provenance_level")
        if incompatible: raise RuntimeError(f"Manifest sorgente incompatibile per {exp['experiment_id']}: {sorted(set(incompatible))}")
        frame = pd.read_csv(source); frames[exp["experiment_id"]] = frame; source_paths[exp["experiment_id"]] = source
    aligned = compare_patient_sets(frames, canonical_ids)
    for eid, frame in aligned.items():
        destination = CENTRAL_PREDICTIONS / f"{eid}.csv"; frame.to_csv(destination, index=False)
        metric_rows.append({"experiment_id": eid, **compute_binary_metrics(frame.y_true, frame.y_score, float(frame.threshold.iloc[0]))})
        exp = registry[eid]; source = source_paths[eid]; source_manifest = PROJECT_ROOT / canonical_test_prediction_paths(exp)["test_predictions_manifest_path"]
        write_prediction_manifest(CENTRAL_PREDICTIONS / f"{eid}.manifest.json", {"experiment_id": eid, "registry_signature": content_signature(REGISTRY_PATH), "source_prediction_manifest_path": str(source_manifest.relative_to(PROJECT_ROOT)), "source_prediction_manifest_signature": content_signature(source_manifest), "source_prediction_file_signature": content_signature(source), "finalists_lock_signature": manifest["lock_signature"], "test_dataset_manifest_signature": content_signature(TEST_DATASET_MANIFEST), "validation_threshold": float(frame.threshold.iloc[0]), "threshold_method": frame.threshold_method.iloc[0], "pipeline_schema_version": 1, "provenance_level": json.loads(source_manifest.read_text()).get("provenance_level", "invalid")}, destination)
    pd.DataFrame(metric_rows).to_csv(OUTPUT_DIR / "final_test_metrics.csv", index=False); (OUTPUT_DIR / "final_test_metrics.json").write_text(strict_json_dumps(metric_rows, indent=2) + "\n")
    dataset_manifest = build_test_dataset_manifest(TEST_CSV, project_root=PROJECT_ROOT, preprocessing={"view": "MLO", "resolution": 512, "grayscale": True, "right_breast_mirrored": True}, include_image_signatures=True)
    (OUTPUT_DIR / "test_dataset_manifest.json").write_text(strict_json_dumps(dataset_manifest, indent=2) + "\n")
    run = {"schema_version": 1, "finalists_lock_signature": manifest["lock_signature"], "test_dataset_signature": value_signature(dataset_manifest), "selection_performed": False, "n_models": len(frames)}
    (OUTPUT_DIR / "final_test_run_manifest.json").write_text(strict_json_dumps(run, indent=2) + "\n")